# Understanding RAG (Retrieval-Augmented Generation) Systems

This notebook explains what RAG systems are, how they work, and why they're important in modern AI applications. We'll implement a RAG system using LangChain with in-memory storage and FAISS vector database, focusing on a recipe and meal planning use case.

## What is RAG?

**Retrieval-Augmented Generation (RAG)** is an AI architecture that combines:

1. **Retrieval**: Finding relevant information from a knowledge base
2. **Generation**: Using that information to generate accurate, contextual responses

RAG enhances large language models (LLMs) by providing them with external knowledge, allowing them to:
- Access domain-specific information not in their training data
- Provide up-to-date information beyond their training cutoff
- Cite specific sources for their responses
- Reduce hallucinations (making up false information)

### Key Components of a RAG System:

1. **Document Collection**: The external knowledge base (documents, databases, etc.)
2. **Document Processing**: Chunking and preparing documents for embedding
3. **Vector Store**: Database that stores vector embeddings of document chunks
4. **Retriever**: Component that finds relevant documents based on a query
5. **Generator**: LLM that creates responses using retrieved information

Let's build a RAG system for recipe and meal planning information!

## Step 1: Install Required Libraries

In [1]:
# Install necessary packages
!pip install langchain langchain-openai langchain-community faiss-cpu python-dotenv sentence-transformers

  Using cached filelock-3.18.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached tokenizers-0.21.1-cp39-abi3-macosx_11_0_arm64.whl.metadata (6.8 kB)
  Using cached safetensors-0.5.3-cp38-abi3-macosx_11_0_arm64.whl.metadata (3.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 763.1 kB/s eta 0:00:001m778.0 kB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 MB 838.0 kB/s eta 0:00:00m eta 0:00:010:00:03
Using cached sympy-1.13.1-py3-none-any.whl (6.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 2.3 MB/s eta 0:00:000:00:0136m0:00:01:01
Using cached safetensors-0.5.3-cp38-abi3-macosx_11_0_arm64.whl (418 kB)
Using cached tokenizers-0.21.1-cp39-abi3-macosx_11_0_arm64.whl (2.7 MB)
Using cached filelock-3.18.0-py3-none-any.whl (16 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 3.6 MB/s eta 0:00:003.7 MB/s eta 0:00:

## Step 2: Import Libraries and Set Up Environment

In [5]:
import os
import json
from dotenv import load_dotenv, find_dotenv
import pandas as pd
# from google.colab import userdata

# LangChain imports
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.schema import Document
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

# Load environment variables (for API keys)
load_dotenv(find_dotenv(), override=True)

# Set OpenAI API key
#os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# If you don't have an API key, you can set it directly (not recommended for production)
api_key = os.environ.get("OPENAI_API_KEY")

## Step 3: Load and Explore Recipe Data

We'll load our recipe data from the JSON file we created.

In [7]:
"""
# prompt: from my google drive read recipes_data.json

# from google.colab import drive
drive.mount('/content/drive')

# Assuming recipes_data.json is in your Google Drive's My Drive folder
file_path = '/content/drive/MyDrive/recipes_data.json'


try:
  with open(file_path, 'r') as f:
    recipe_data = json.load(f)
    # Now you can work with the 'data' variable which holds the JSON data
    print("Successfully loaded JSON data from Google Drive.")

except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
except json.JSONDecodeError:
    print(f"Error: Invalid JSON format in {file_path}")
except Exception as e:
  print(f"An unexpected error occurred: {e}")
"""

'\n# prompt: from my google drive read recipes_data.json\n\n# from google.colab import drive\ndrive.mount(\'/content/drive\')\n\n# Assuming recipes_data.json is in your Google Drive\'s My Drive folder\nfile_path = \'/content/drive/MyDrive/recipes_data.json\'\n\n\ntry:\n  with open(file_path, \'r\') as f:\n    recipe_data = json.load(f)\n    # Now you can work with the \'data\' variable which holds the JSON data\n    print("Successfully loaded JSON data from Google Drive.")\n\nexcept FileNotFoundError:\n    print(f"Error: File not found at {file_path}")\nexcept json.JSONDecodeError:\n    print(f"Error: Invalid JSON format in {file_path}")\nexcept Exception as e:\n  print(f"An unexpected error occurred: {e}")\n'

In [9]:
import json

# Carica il file JSON dalla directory corrente
with open('recipes_data.json', 'r') as file:
    recipe_data = json.load(file)

## Step 4: Prepare Documents for RAG

To build our RAG system, we need to:
1. Convert our JSON data into document format
2. Split documents into chunks for better retrieval
3. Create embeddings and store them in a vector database

In [10]:
# Convert recipes to document format
documents = []

for recipe in recipe_data['recipes']:
    # Format ingredients as a list
    ingredients_text = "\nIngredients:\n" + "\n".join([f"- {ingredient}" for ingredient in recipe['ingredients']])

    # Create a comprehensive text representation of the recipe
    content = f"""Recipe: {recipe['title']}
Cuisine: {recipe['cuisine']}
Meal Type: {recipe['mealType']}
Preparation Time: {recipe['prepTime']}
Cooking Time: {recipe['cookTime']}
Servings: {recipe['servings']}
Dietary Information: {', '.join(recipe['dietaryInfo'])}
{ingredients_text}

Instructions:
{recipe['instructions']}
"""

    # Create a document with metadata
    doc = Document(
        page_content=content,
        metadata={
            "title": recipe['title'],
            "cuisine": recipe['cuisine'],
            "meal_type": recipe['mealType'],
            "id": recipe['id']
        }
    )

    documents.append(doc)

print(f"Created {len(documents)} documents")
print("\nSample document:")
print(documents[0].page_content[:300] + "...")
print("\nMetadata:", documents[0].metadata)

Created 10 documents

Sample document:
Recipe: Classic Spaghetti Bolognese
Cuisine: Italian
Meal Type: Dinner
Preparation Time: 15 minutes
Cooking Time: 45 minutes
Servings: 4
Dietary Information: High protein, Contains gluten, Contains dairy

Ingredients:
- 500g ground beef
- 1 onion, finely chopped
- 2 garlic cloves, minced
- 2 carrots...

Metadata: {'title': 'Classic Spaghetti Bolognese', 'cuisine': 'Italian', 'meal_type': 'Dinner', 'id': 1}


### Split Documents into Chunks

For better retrieval, we'll split our documents into smaller chunks. This helps the system find the most relevant pieces of information.

In [11]:
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    is_separator_regex=False,
)

# Split documents into chunks
chunks = text_splitter.split_documents(documents)

print(f"Split {len(documents)} documents into {len(chunks)} chunks")
print("\nSample chunk:")
print(chunks[0].page_content)
print("\nChunk metadata:", chunks[0].metadata)

Split 10 documents into 23 chunks

Sample chunk:
Recipe: Classic Spaghetti Bolognese
Cuisine: Italian
Meal Type: Dinner
Preparation Time: 15 minutes
Cooking Time: 45 minutes
Servings: 4
Dietary Information: High protein, Contains gluten, Contains dairy

Chunk metadata: {'title': 'Classic Spaghetti Bolognese', 'cuisine': 'Italian', 'meal_type': 'Dinner', 'id': 1}


## Step 5: Create Vector Store with FAISS

Now we'll create embeddings for our document chunks and store them in a FAISS vector database. FAISS (Facebook AI Similarity Search) is an efficient library for similarity search and clustering of dense vectors.

In [12]:
from sentence_transformers import SentenceTransformer
from langchain.embeddings import HuggingFaceEmbeddings


# Use a smaller, more efficient embedding model
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

/var/folders/tl/zb5lzmw1379d6f96zdj440lm0000gn/T/ipykernel_34430/2707869699.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
# Create FAISS vector store from documents
vectorstore = FAISS.from_documents(chunks, embeddings)

print(f"Created vector store with {len(chunks)} embedded chunks")

Created vector store with 23 embedded chunks


### Test Simple Retrieval

Let's test our vector store by retrieving documents related to a simple query.

In [14]:
# Test retrieval with a simple query
query = "vegetarian breakfast recipes"
docs = vectorstore.similarity_search(query, k=2)

print(f"Query: '{query}'")
print(f"Retrieved {len(docs)} documents\n")

for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(f"Title: {doc.metadata['title']}")
    print(f"Content: {doc.page_content[:200]}...\n")

Query: 'vegetarian breakfast recipes'
Retrieved 2 documents

Document 1:
Title: Vegetable Stir Fry
Content: Recipe: Vegetable Stir Fry
Cuisine: Asian
Meal Type: Dinner
Preparation Time: 15 minutes
Cooking Time: 10 minutes
Servings: 2
Dietary Information: Vegetarian, Can be made vegan, Gluten-free with tamar...

Document 2:
Title: Berry Smoothie Bowl
Content: Recipe: Berry Smoothie Bowl
Cuisine: Modern/Health
Meal Type: Breakfast
Preparation Time: 10 minutes
Cooking Time: 0 minutes
Servings: 1
Dietary Information: Vegetarian, Contains dairy, Can be made ve...



### Test Simple Prompts Without RAG

Let's test simple prompts without using RAG System.

In [15]:
# Initialize the language model
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

In [16]:
normal_template_wo_rag = """
You are a helpful cooking assistant that provides information about recipes and meal planning.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Question: {question}

Answer:
"""

# Create a prompt from the template
normal_prompt_wo_rag = ChatPromptTemplate.from_template(normal_template_wo_rag)

normal_chain_wo_rag = (
    {"question": RunnablePassthrough()}
    | normal_prompt_wo_rag
    | llm
    | StrOutputParser()
)

In [17]:
# Test query 1: Simple recipe question
query1 = "What ingredients do I need for a vegetable stir fry?"
response1_wo_rag = normal_chain_wo_rag.invoke(query1)

print(f"Query: {query1}")
print(f"Response:\n{response1_wo_rag}\n")

Query: What ingredients do I need for a vegetable stir fry?
Response:
For a basic vegetable stir fry, you will need a variety of vegetables such as bell peppers, broccoli, carrots, snap peas, and mushrooms. You will also need garlic, ginger, soy sauce, and a cooking oil such as sesame oil or vegetable oil. Additionally, you can add protein sources like tofu, chicken, or shrimp if desired. Feel free to customize the ingredients based on your preferences!



In [18]:
# Test query 2: Meal planning question
query2 = "Suggest a healthy breakfast recipe that's quick to make"
response2_wo_rag = normal_chain_wo_rag.invoke(query2)

print(f"Query: {query2}")
print(f"Response:\n{response2_wo_rag}\n")

Query: Suggest a healthy breakfast recipe that's quick to make
Response:
One option could be overnight oats. Simply mix oats with your choice of milk or yogurt, add in some chia seeds or flaxseeds for extra nutrition, and let it sit in the fridge overnight. In the morning, you can top it with fruits, nuts, or a drizzle of honey for added flavor. It's a quick and easy breakfast that can be prepared the night before.



In [19]:
# Test query 3: Dietary restriction question
query3 = "What vegetarian dinner options do you have?"
response3_wo_rag = normal_chain_wo_rag.invoke(query3)

print(f"Query: {query3}")
print(f"Response:\n{response3_wo_rag}\n")

Query: What vegetarian dinner options do you have?
Response:
Some vegetarian dinner options you could consider are:
- Vegetable stir-fry with tofu
- Lentil soup
- Quinoa salad with roasted vegetables
- Eggplant parmesan
- Chickpea curry
- Stuffed bell peppers
- Mushroom risotto
- Veggie tacos
- Caprese pasta salad
- Spinach and feta stuffed portobello mushrooms

Please let me know if you need more information or specific recipes for any of these options!



## Step 6: Build a Complete RAG Pipeline

Now we'll build a complete RAG pipeline using LangChain's components:
1. Retrieve relevant documents based on a query
2. Format the retrieved information
3. Generate a response using an LLM

In [20]:
# Create a retriever from the vector store
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Define a template for generating responses
template = """
You are a helpful cooking assistant that provides information about recipes and meal planning.
Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context:
{context}

Question: {question}

Answer:
"""

# Create a prompt from the template
prompt = ChatPromptTemplate.from_template(template)

# Define a function to format documents
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

# Build the RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## Step 7: Test the RAG System

Let's test our RAG system with various queries related to recipes and meal planning.

In [21]:
# Test query 1: Simple recipe question: "What ingredients do I need for a vegetable stir fry?"
response1 = rag_chain.invoke(query1)

print(f"Query: {query1}")
print(f"Response:\n{response1}\n")

Query: What ingredients do I need for a vegetable stir fry?
Response:
- 200g tofu, cubed
- 1 bell pepper, sliced
- 1 carrot, julienned
- 100g broccoli florets
- 100g snow peas
- 2 tbsp vegetable oil
- 2 garlic cloves, minced
- 1 inch ginger, grated
- 3 tbsp soy sauce
- 1 tbsp sesame oil
- 1 tbsp honey



In [22]:
print(f"Response:\n{response1_wo_rag}\n")

Response:
For a basic vegetable stir fry, you will need a variety of vegetables such as bell peppers, broccoli, carrots, snap peas, and mushrooms. You will also need garlic, ginger, soy sauce, and a cooking oil such as sesame oil or vegetable oil. Additionally, you can add protein sources like tofu, chicken, or shrimp if desired. Feel free to customize the ingredients based on your preferences!



In [23]:
# Test query 2: Meal planning question: "Suggest a healthy breakfast recipe that's quick to make"
response2 = rag_chain.invoke(query2)

print(f"Query: {query2}")
print(f"Response:\n{response2}\n")

Query: Suggest a healthy breakfast recipe that's quick to make
Response:
One healthy and quick breakfast recipe you can try is Overnight Oats. Just mix together 1/2 cup of rolled oats, 1/2 cup of milk (or a dairy-free alternative), 1 tablespoon of chia seeds, 1/2 teaspoon of vanilla extract, and a pinch of cinnamon in a jar or container. Let it sit in the fridge overnight, and in the morning, top it with your favorite fruits, nuts, or seeds for added flavor and nutrition. Enjoy!



In [24]:
print(f"Response:\n{response2_wo_rag}\n")

Response:
One option could be overnight oats. Simply mix oats with your choice of milk or yogurt, add in some chia seeds or flaxseeds for extra nutrition, and let it sit in the fridge overnight. In the morning, you can top it with fruits, nuts, or a drizzle of honey for added flavor. It's a quick and easy breakfast that can be prepared the night before.



In [25]:
# Test query 3: Dietary restriction question
query3 = "What vegetarian dinner options do you have?"
response3 = rag_chain.invoke(query3)

print(f"Query: {query3}")
print(f"Response:\n{response3}\n")

Query: What vegetarian dinner options do you have?
Response:
Some vegetarian dinner options you can consider are:
1. Vegetable Stir Fry
2. Quinoa Salad with Roasted Vegetables



In [26]:
print(f"Response:\n{response3_wo_rag}\n")

Response:
Some vegetarian dinner options you could consider are:
- Vegetable stir-fry with tofu
- Lentil soup
- Quinoa salad with roasted vegetables
- Eggplant parmesan
- Chickpea curry
- Stuffed bell peppers
- Mushroom risotto
- Veggie tacos
- Caprese pasta salad
- Spinach and feta stuffed portobello mushrooms

Please let me know if you need more information or specific recipes for any of these options!



## Step 8: Advanced RAG Features - Adding Memory

Let's enhance our RAG system by adding memory to maintain context across multiple queries.

In [27]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain

# Initialize memory
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# Create a conversational retrieval chain
conversation_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    verbose=True
)

/var/folders/tl/zb5lzmw1379d6f96zdj440lm0000gn/T/ipykernel_34430/634010990.py:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


In [28]:
# Test conversation with memory
print("Starting conversation with memory...\n")

# First query
query1 = "What ingredients do I need for a vegetable stir fry?"
result1 = conversation_chain.invoke({"question": query1})
print(f"Query: {query1}")
print(f"Response: {result1['answer']}\n")

# Follow-up query that relies on conversation context
query2 = "How long does it take to cook?"
result2 = conversation_chain.invoke({"question": query2})
print(f"Query: {query2}")
print(f"Response: {result2['answer']}\n")

# Another follow-up query
query3 = "Can I make it vegan?"
result3 = conversation_chain.invoke({"question": query3})
print(f"Query: {query3}")
print(f"Response: {result3['answer']}")

Starting conversation with memory...



> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: Use the following pieces of context to answer the user's question. 
If you don't know the answer, just say that you don't know, don't try to make up an answer.
----------------
Recipe: Vegetable Stir Fry
Cuisine: Asian
Meal Type: Dinner
Preparation Time: 15 minutes
Cooking Time: 10 minutes
Servings: 2
Dietary Information: Vegetarian, Can be made vegan, Gluten-free with tamari

Ingredients:
- 200g tofu, cubed
- 1 bell pepper, sliced
- 1 carrot, julienned
- 100g broccoli florets
- 100g snow peas
- 2 tbsp vegetable oil
- 2 garlic cloves, minced
- 1 inch ginger, grated
- 3 tbsp soy sauce
- 1 tbsp sesame oil
- 1 tbsp honey
- 2 cups cooked rice

Ingredients:
- 1 cup quinoa
- 2 cups vegetable stock
- 1 zucchini, diced
- 1 eggplant, diced
- 1 red bell pepper, diced
- 1 red onion, diced
- 2 tbsp olive oil
- 1 lemon, juiced
- 1/4 cup fresh herbs 

## Step 9: Building a Meal Planner with RAG

Let's create a more complex application: a meal planner that uses our RAG system to suggest meals based on dietary preferences and available ingredients.

In [ ]:
# Define a specialized prompt for meal planning
meal_planner_template = """
You are a helpful meal planning assistant. Based on the recipe information provided and the user's preferences,
create a meal plan as requested. Use only the recipes mentioned in the context or variations of them.

Context (Recipe Information):
{context}

User Request: {question}

Provide a detailed meal plan with recipe suggestions, preparation tips, and any modifications needed to meet the user's requirements.
Format your response in a clear, organized way with headings and bullet points as appropriate.
"""

meal_planner_prompt = ChatPromptTemplate.from_template(meal_planner_template)

# Build the meal planner RAG chain
meal_planner_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | meal_planner_prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# Test the meal planner with a complex request
meal_plan_request = """I need a 3-day vegetarian meal plan with breakfast, lunch, and dinner.
I prefer high-protein options and I have these ingredients available: quinoa, tofu, eggs, yogurt,
various vegetables, and fruits. I don't like mushrooms."""

meal_plan = meal_planner_chain.invoke(meal_plan_request)

print("Meal Plan Request:")
print(meal_plan_request)
print("\nGenerated Meal Plan:")
print(meal_plan)

Meal Plan Request:
I need a 3-day vegetarian meal plan with breakfast, lunch, and dinner. 
I prefer high-protein options and I have these ingredients available: quinoa, tofu, eggs, yogurt, 
various vegetables, and fruits. I don't like mushrooms.

Generated Meal Plan:
**Day 1:**

**Breakfast:**
- Quinoa Breakfast Bowl
  - Cook quinoa according to package instructions.
  - Top with yogurt, sliced fruits, and a drizzle of honey for sweetness.
  - Optional: Add nuts or seeds for extra protein.

**Lunch:**
- Quinoa Salad with Roasted Vegetables (from provided recipe)
  - Prepare the quinoa salad as per the recipe.
  - Serve with a side of fresh fruit for a balanced meal.

**Dinner:**
- Tofu Stir Fry
  - In a pan, heat vegetable oil and sauté tofu until golden brown.
  - Add sliced bell pepper, julienned carrot, broccoli florets, and snow peas.
  - Mix in minced garlic, grated ginger, soy sauce, sesame oil, and honey.
  - Serve over cooked rice for a satisfying meal.

**Day 2:**

**Breakfast

## Step 10: Understanding RAG System Components

Let's break down the key components of our RAG system and understand their roles:

### 1. Document Processing

We processed our recipe data by:
- Converting JSON to document format
- Adding metadata for better retrieval
- Splitting into smaller chunks for more precise retrieval

### 2. Vector Store (FAISS)

FAISS provides:
- Efficient storage of document embeddings
- Fast similarity search capabilities
- In-memory storage for quick access

### 3. Retriever

The retriever:
- Takes a user query
- Converts it to an embedding
- Finds similar documents in the vector store
- Returns the most relevant documents

### 4. Prompt Engineering

Our prompts:
- Format retrieved context and user questions
- Guide the LLM to generate appropriate responses
- Can be specialized for different tasks (general QA vs. meal planning)

### 5. Language Model (LLM)

The LLM:
- Generates responses based on the prompt and retrieved context
- Synthesizes information from multiple sources
- Formats responses according to instructions

### 6. Memory (Optional)

Memory components:
- Store conversation history
- Provide context for follow-up questions
- Enable more natural, flowing conversations

## Step 11: Evaluating RAG System Performance

Let's implement a simple evaluation to assess how well our RAG system performs on different types of queries.

In [29]:
# Define test queries covering different aspects
test_queries = [
    "What's a good breakfast recipe that contains eggs?",
    "I need a vegetarian dinner recipe that's high in protein",
    "How do I make a chocolate dessert?",
    "What recipes can I make with chicken?",
    "I need a quick lunch recipe that takes less than 15 minutes to prepare"
]

# Run evaluation
print("RAG System Evaluation\n")
print("-" * 50)

for i, query in enumerate(test_queries):
    print(f"Test Query {i+1}: {query}")

    # Get retrieved documents
    docs = retriever.get_relevant_documents(query)

    # Print retrieved document titles
    print("Retrieved documents:")
    for j, doc in enumerate(docs):
        print(f"  {j+1}. {doc.metadata['title']}")

    # Generate response
    response = rag_chain.invoke(query)
    print(f"\nResponse: {response[:200]}...\n")
    print("-" * 50)

RAG System Evaluation

--------------------------------------------------
Test Query 1: What's a good breakfast recipe that contains eggs?


/var/folders/tl/zb5lzmw1379d6f96zdj440lm0000gn/T/ipykernel_34430/3139935098.py:18: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents(query)


Retrieved documents:
  1. Avocado Toast with Poached Egg
  2. Chicken Caesar Salad

Response: A good breakfast recipe that contains eggs is the Avocado Toast with Poached Egg....

--------------------------------------------------
Test Query 2: I need a vegetarian dinner recipe that's high in protein
Retrieved documents:
  1. Vegetable Stir Fry
  2. Beef and Vegetable Stew

Response: One option for a vegetarian dinner recipe that's high in protein is a Lentil and Chickpea Curry. Lentils and chickpeas are both excellent sources of plant-based protein. Would you like the recipe for ...

--------------------------------------------------
Test Query 3: How do I make a chocolate dessert?
Retrieved documents:
  1. Chocolate Chip Cookies
  2. Berry Smoothie Bowl

Response: To make a chocolate dessert, you can try making chocolate chip cookies. Preheat the oven to 350°F. Cream butter and sugars until smooth. Beat in eggs one at a time, then stir in vanilla. Dissolve baki...

------------------

## Step 12: Advantages and Limitations of RAG Systems

### Advantages of RAG:

1. **Up-to-date information**: RAG can access the latest information, unlike LLMs limited by training data cutoffs
2. **Domain specificity**: Can be tailored to specific domains with specialized knowledge
3. **Reduced hallucinations**: By grounding responses in retrieved documents
4. **Transparency**: Can cite sources for information provided
5. **Flexibility**: Can update knowledge base without retraining the model

### Limitations of RAG:

1. **Retrieval quality dependency**: System is only as good as its retrieval mechanism
2. **Context window limitations**: Can only use a limited amount of retrieved context
3. **Computational overhead**: Additional processing for retrieval and embedding
4. **Knowledge gaps**: Limited to information in the knowledge base
5. **Complex reasoning challenges**: May struggle with queries requiring synthesis across many documents

## Step 13: Best Practices for RAG Systems

1. **Effective chunking**: Choose appropriate chunk sizes for your content
2. **Metadata enrichment**: Add useful metadata to improve retrieval relevance
3. **Prompt engineering**: Craft prompts that guide the LLM effectively
4. **Hybrid retrieval**: Combine semantic search with keyword search for better results
5. **Evaluation**: Regularly test system performance with diverse queries
6. **Feedback loops**: Incorporate user feedback to improve the system
7. **Knowledge base maintenance**: Keep your knowledge base updated and well-organized

## Conclusion

In this notebook, we've explored RAG (Retrieval-Augmented Generation) systems by building a recipe and meal planning assistant. We've covered:

1. The fundamental components of RAG systems
2. How to process documents and create embeddings
3. Using FAISS for efficient vector storage and retrieval
4. Building RAG pipelines with LangChain
5. Adding conversation memory for context-aware responses
6. Creating specialized applications like meal planners
7. Evaluating RAG system performance
8. Understanding the advantages and limitations of RAG

RAG systems represent a powerful approach to enhancing LLMs with external knowledge, making them more accurate, transparent, and useful for domain-specific applications.